# Integer Linear Programming (ILP) Solutions

This notebook contains the Python (`PuLP 4.0`) code and verified solutions for all Integer Linear Programming, Mixed-Integer Linear Programming (MILP), and Sensitivity Analysis examples presented in the `Integer_Linear_Programming.tex` slide deck for **BUS 104**.

### 1. Simple Pure Integer Linear Program & Branch-and-Bound

**Problem Formulation:**
$$\max z = 3x_1 + 2x_2$$
subject to:
$$\begin{aligned}
2x_1 + x_2 &\le 7 \quad (C_1) \\
-x_1 + x_2 &\le 3 \quad (C_2) \\
x_1, x_2 &\in \mathbb{Z}_{\ge 0}
\end{aligned}$$

The cell below runs the exact `PuLP 4.0` code from **Section 1** of the slides (`Simple_Pure_ILP`).

In [1]:
import pulp

# 1. Initialize Pure ILP Model (PuLP 4.0 API)
prob = pulp.LpProblem("Simple_Pure_ILP", pulp.LpMaximize)

# 2. Integer Decision Variables: x1, x2 in Z_{>= 0}
x1 = prob.add_variable("x1", lowBound=0, cat=pulp.LpInteger)
x2 = prob.add_variable("x2", lowBound=0, cat=pulp.LpInteger)

# 3. Objective Function
prob += 3 * x1 + 2 * x2, "Total_Profit"

# 4. Resource Constraints
prob += 2 * x1 + x2 <= 7, "C1"
prob += -x1 + x2 <= 3, "C2"

# 5. Solve and inspect Result
res = prob.solve()
print(f"Status: {res.status.name}")
print(f"Optimal z* = {pulp.value(prob.objective):.2f}")
print(f"x1* = {x1.varValue}, x2* = {x2.varValue}")

Welcome to the CBC MILP Solver 
Version: 2.10.13 
Build Date: Mar 11 2026 

command line - cbc /var/folders/x2/y620t7fd5dl06n7bq9jsb9qw0000gp/T/fecd4965585c4789a6dd8d6acf905fc7-pulp.mps -timeMode elapsed -solve -printingOptions all -solution /var/folders/x2/y620t7fd5dl06n7bq9jsb9qw0000gp/T/fecd4965585c4789a6dd8d6acf905fc7-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 7 COLUMNS
At line 18 RHS
At line 21 BOUNDS
At line 24 ENDATA
Problem MODEL has 2 rows, 2 columns and 4 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is -12.6667 - 0.00 seconds
Cgl0004I processed model has 2 rows, 2 columns (2 integer (0 of which binary)) and 4 elements
Cutoff increment increased from 1e-05 to 0.9999
Cbc0012I Integer solution of -11 found by DiveCoefficient after 0 iterations and 0 nodes (0.00 seconds)
Cbc0012I Integer solution of -12 found by DiveCoefficient after 1 iterations and 0 nodes (0.00 sec

#### 1b. Comparing the Continuous LP Relaxation vs. Pure ILP (Why Naive Rounding Fails)

In the continuous LP relaxation ($x_1, x_2 \ge 0$), the optimal solution is $(x_1^*, x_2^*) = (4/3, 13/3) \approx (1.33, 4.33)$ with $z_{LP}^* = 12.67$. Rounding down to $(1, 4)$ yields $z = 11.00$ (suboptimal), while rounding to $(1, 5)$, $(2, 4)$, or $(2, 5)$ violates the constraints. Branch-and-Bound finds the true integer optimum $(2, 3)$ with $z^* = 12.00$.

In [2]:
import pulp

# Solve the Continuous LP Relaxation
lp_relax = pulp.LpProblem("LP_Relaxation", pulp.LpMaximize)
x1_c = lp_relax.add_variable("x1", lowBound=0, cat=pulp.LpContinuous)
x2_c = lp_relax.add_variable("x2", lowBound=0, cat=pulp.LpContinuous)

lp_relax += 3 * x1_c + 2 * x2_c, "Total_Profit"
lp_relax += 2 * x1_c + x2_c <= 7, "C1"
lp_relax += -x1_c + x2_c <= 3, "C2"

lp_relax.solve(pulp.COIN_CMD(msg=False))
print(f"LP Relaxation Optimum : x1* = {x1_c.varValue:.4f}, x2* = {x2_c.varValue:.4f}, z_LP* = {pulp.value(lp_relax.objective):.4f}")
print(f"Naive Rounded (1, 4)  : x1  = 1.0000, x2  = 4.0000, z     = {3*1 + 2*4:.4f} (Suboptimal)")
print(f"True Pure ILP Optimum : x1* = 2.0000, x2* = 3.0000, z_ILP*= {3*2 + 2*3:.4f} (Optimal)")

LP Relaxation Optimum : x1* = 1.3333, x2* = 4.3333, z_LP* = 12.6667
Naive Rounded (1, 4)  : x1  = 1.0000, x2  = 4.0000, z     = 11.0000 (Suboptimal)
True Pure ILP Optimum : x1* = 2.0000, x2* = 3.0000, z_ILP*= 12.0000 (Optimal)


### 2. The 0-1 Knapsack Problem

**Problem Context:**
A hiker can carry at most $W = 10\text{ kg}$ and must choose which items to pack ($x_i \in \{0, 1\}$):
- **Laptop:** Value $= \$40$, Weight $= 4\text{ kg}$ (Ratio $= \$10.00/\text{kg}$)
- **Gold:** Value $= \$50$, Weight $= 3\text{ kg}$ (Ratio $= \$16.67/\text{kg}$)
- **Silver:** Value $= \$30$, Weight $= 6\text{ kg}$ (Ratio $= \$5.00/\text{kg}$)
- **Books:** Value $= \$10$, Weight $= 5\text{ kg}$ (Ratio $= \$2.00/\text{kg}$)

The cell below runs the exact `PuLP 4.0` code from **Section 2** of the slides (`Knapsack_0_1`).

In [3]:
import pulp

items = ["Laptop", "Gold", "Silver", "Books"]
val = {"Laptop": 40, "Gold": 50, "Silver": 30, "Books": 10}
wt  = {"Laptop": 4,  "Gold": 3,  "Silver": 6,  "Books": 5}

prob = pulp.LpProblem("Knapsack_0_1", pulp.LpMaximize)

# Binary variables x[i] in {0, 1} using PuLP 4.0 API
x = prob.add_variable_dicts("x", items, cat=pulp.LpBinary)

# Objective & Weight Capacity Constraint
prob += pulp.lpSum(val[i] * x[i] for i in items), "Total_Value"
prob += pulp.lpSum(wt[i] * x[i] for i in items) <= 10, "Max_Weight"

res = prob.solve()
z_opt = pulp.value(prob.objective)
print(f"Status: {res.status.name}, Optimal Value = ${z_opt:.0f}")
for i in items:
    if x[i].varValue > 0.5:
        print(f"  Take {i}: weight={wt[i]} kg, value=${val[i]}")

Welcome to the CBC MILP Solver 
Version: 2.10.13 
Build Date: Mar 11 2026 

command line - cbc /var/folders/x2/y620t7fd5dl06n7bq9jsb9qw0000gp/T/230bb4812e13441e8e8eed75a137f55f-pulp.mps -timeMode elapsed -solve -printingOptions all -solution /var/folders/x2/y620t7fd5dl06n7bq9jsb9qw0000gp/T/230bb4812e13441e8e8eed75a137f55f-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 6 COLUMNS
At line 23 RHS
At line 25 BOUNDS
At line 30 ENDATA
Problem MODEL has 1 rows, 4 columns and 4 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is -105 - 0.00 seconds
Cgl0004I processed model has 1 rows, 4 columns (4 integer (4 of which binary)) and 4 elements
Cutoff increment increased from 1e-05 to 9.9999
Cbc0038I Initial state - 1 integers unsatisfied sum - 0.5
Cbc0038I Solution found of -90
Cbc0038I Before mini branch and bound, 3 integers at bound fixed and 0 continuous
Cbc0038I Full problem 1 rows 4 colu

#### 2b. When Does the Greedy Heuristic Fail? (Counterexample from Slides)

Consider a knapsack with capacity $W = 6\text{ kg}$ and three items:
- **Item 1:** Value $= \$32$, Weight $= 4\text{ kg}$ (Ratio $= \$8.00/\text{kg}$)
- **Item 2:** Value $= \$22$, Weight $= 3\text{ kg}$ (Ratio $= \$7.33/\text{kg}$)
- **Item 3:** Value $= \$22$, Weight $= 3\text{ kg}$ (Ratio $= \$7.33/\text{kg}$)

The greedy heuristic selects **Item 1** first (highest ratio $\$8.00/\text{kg}$), leaving $2\text{ kg}$ of unused capacity and earning **$z_{\text{greedy}} = \$32$**. The 0-1 ILP solver selects **Items 2 and 3**, filling all $6\text{ kg}$ and earning **$z^* = \$44$**.

In [4]:
import pulp

items_ce = ["Item_1", "Item_2", "Item_3"]
val_ce = {"Item_1": 32, "Item_2": 22, "Item_3": 22}
wt_ce  = {"Item_1": 4,  "Item_2": 3,  "Item_3": 3}

prob_ce = pulp.LpProblem("Knapsack_Greedy_Counterexample", pulp.LpMaximize)
x_ce = prob_ce.add_variable_dicts("x", items_ce, cat=pulp.LpBinary)

prob_ce += pulp.lpSum(val_ce[i] * x_ce[i] for i in items_ce), "Total_Value"
prob_ce += pulp.lpSum(wt_ce[i] * x_ce[i] for i in items_ce) <= 6, "Capacity_6kg"

prob_ce.solve(pulp.COIN_CMD(msg=False))
print(f"Greedy Heuristic Choice : ['Item_1']          -> Weight = 4 kg, Value = $32")
print(f"Optimal 0-1 ILP Choice  : {[i for i in items_ce if x_ce[i].varValue > 0.5]} -> Weight = 6 kg, Value = ${pulp.value(prob_ce.objective):.0f}")

Greedy Heuristic Choice : ['Item_1']          -> Weight = 4 kg, Value = $32
Optimal 0-1 ILP Choice  : ['Item_2', 'Item_3'] -> Weight = 6 kg, Value = $44


### 3. 0-1 Knapsack with Logical Constraints

**Logical Conditions Added to the 4-Item Knapsack Problem:**
1. **Mutual Exclusion:** Cannot take both Laptop and Gold ($x_{\text{Laptop}} + x_{\text{Gold}} \le 1$).
2. **Implication (If-Then):** If Gold is taken, Books must also be taken ($x_{\text{Gold}} - x_{\text{Books}} \le 0$).

The cell below runs the exact `PuLP 4.0` code from **Section 3** of the slides (`Knapsack_Logical`).

In [5]:
import pulp

items = ["Laptop", "Gold", "Silver", "Books"]
val = {"Laptop": 40, "Gold": 50, "Silver": 30, "Books": 10}
wt  = {"Laptop": 4,  "Gold": 3,  "Silver": 6,  "Books": 5}

prob = pulp.LpProblem("Knapsack_Logical", pulp.LpMaximize)
x = prob.add_variable_dicts("x", items, cat=pulp.LpBinary)

prob += pulp.lpSum(val[i] * x[i] for i in items), "Total_Value"
prob += pulp.lpSum(wt[i] * x[i] for i in items) <= 10, "Max_Weight"

# Rule 1 (Mutual Exclusion): Cannot take both Laptop and Gold
prob += x["Laptop"] + x["Gold"] <= 1, "Mutually_Exclusive"

# Rule 2 (Implication): If Gold is taken, Books must be taken
prob += x["Gold"] - x["Books"] <= 0, "Gold_Implies_Books"

res = prob.solve()
z_opt = pulp.value(prob.objective)
print(f"Status: {res.status.name}, Optimal Value = ${z_opt:.0f}")
print("Selected:", [i for i in items if x[i].varValue > 0.5])

Welcome to the CBC MILP Solver 
Version: 2.10.13 
Build Date: Mar 11 2026 

command line - cbc /var/folders/x2/y620t7fd5dl06n7bq9jsb9qw0000gp/T/6e6eae66242f488d87f1d23284a4dcbb-pulp.mps -timeMode elapsed -solve -printingOptions all -solution /var/folders/x2/y620t7fd5dl06n7bq9jsb9qw0000gp/T/6e6eae66242f488d87f1d23284a4dcbb-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 8 COLUMNS
At line 29 RHS
At line 33 BOUNDS
At line 38 ENDATA
Problem MODEL has 3 rows, 4 columns and 8 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is -70 - 0.00 seconds
Cgl0004I processed model has 3 rows, 4 columns (4 integer (4 of which binary)) and 8 elements
Cutoff increment increased from 1e-05 to 9.9999
Cbc0038I Initial state - 0 integers unsatisfied sum - 1.11022e-16
Cbc0038I Solution found of -70
Cbc0038I Before mini branch and bound, 4 integers at bound fixed and 0 continuous
Cbc0038I Mini branch and bou

### 4. Fixed Setup Cost Problem (Mixed-Integer Linear Program)

**Problem Formulation:**
$$\max Z = 5x_1 + 3x_2 - 10y$$
subject to:
$$\begin{aligned}
x_1 + x_2 &\le 10 \quad (\text{Resource A}) \\
2x_1 + x_2 &\le 16 \quad (\text{Resource B}) \\
x_2 &\le 7 \quad (\text{Resource C}) \\
x_1 + x_2 - 10y &\le 0 \quad (\text{Big-}M \text{ Linking Constraint, } M = 10) \\
x_1, x_2 &\ge 0, \quad y \in \{0, 1\}
\end{aligned}$$

The cell below runs the exact `PuLP 4.0` code from **Section 4** of the slides (`Fixed_Setup_Cost_MILP`).

In [6]:
import pulp

prob = pulp.LpProblem("Fixed_Setup_Cost_MILP", pulp.LpMaximize)

# Continuous production variables and binary setup variable
x1 = prob.add_variable("x1", lowBound=0, cat=pulp.LpContinuous)
x2 = prob.add_variable("x2", lowBound=0, cat=pulp.LpContinuous)
y  = prob.add_variable("y", cat=pulp.LpBinary)

# Objective: Gross Profit - Fixed Setup Cost
prob += 5 * x1 + 3 * x2 - 10 * y, "Net_Profit"

# Resource constraints + Big-M linking constraint (M = 10)
prob += x1 + x2 <= 10, "Resource_A"
prob += 2 * x1 + x2 <= 16, "Resource_B"
prob += x2 <= 7, "Resource_C"
prob += x1 + x2 - 10 * y <= 0, "Big_M_Link"

res = prob.solve()
z_opt = pulp.value(prob.objective)
print(f"Status: {res.status.name}, Net Profit Z* = ${z_opt:.2f}")
print(f"Activate y*={int(y.varValue)}, x1*={x1.varValue}, x2*={x2.varValue}")

Welcome to the CBC MILP Solver 
Version: 2.10.13 
Build Date: Mar 11 2026 

command line - cbc /var/folders/x2/y620t7fd5dl06n7bq9jsb9qw0000gp/T/e33a4f7fcdde4e5ca4b463c763493d16-pulp.mps -timeMode elapsed -solve -printingOptions all -solution /var/folders/x2/y620t7fd5dl06n7bq9jsb9qw0000gp/T/e33a4f7fcdde4e5ca4b463c763493d16-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 9 COLUMNS
At line 23 RHS
At line 28 BOUNDS
At line 30 ENDATA
Problem MODEL has 4 rows, 3 columns and 8 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is -32 - 0.00 seconds
Cgl0003I 0 fixed, 0 tightened bounds, 2 strengthened rows, 0 substitutions
Cgl0004I processed model has 2 rows, 3 columns (1 integer (1 of which binary)) and 6 elements
Cbc0038I Initial state - 0 integers unsatisfied sum - 0
Cbc0038I Solution found of -32
Cbc0038I Relaxing continuous gives -32
Cbc0038I Before mini branch and bound, 1 integers at b

### 5. Minimum Purchase Quantity (MOQ) Problem

**Problem Formulation:**
A firm may place an order with a supplier ($y \in \{0, 1\}$) with a fixed setup fee of $\$12$ and a **Minimum Order Quantity (MOQ)** of $6$ total units whenever $y = 1$:
$$\max Z = 7x_1 + 5x_2 - 12y$$
subject to:
$$\begin{aligned}
2x_1 + x_2 &\le 16 \quad (\text{Assembly Hours}) \\
x_1 + 2x_2 &\le 18 \quad (\text{Material Limit}) \\
x_2 &\le 8 \quad (\text{Demand Cap for B}) \\
x_1 + x_2 - 6y &\ge 0 \quad (\text{Minimum Order Quantity: } x_1 + x_2 \ge 6 \text{ if } y = 1) \\
x_1 - 18y &\le 0 \quad (\text{Upper Bound Link for } x_1) \\
x_2 - 18y &\le 0 \quad (\text{Upper Bound Link for } x_2) \\
x_1, x_2 &\in \mathbb{Z}_{\ge 0}, \quad y \in \{0, 1\}
\end{aligned}$$

The cell below runs the exact `PuLP 4.0` code from **Section 5** of the slides (`Supplier_Order_MOQ`).

In [7]:
import pulp

prob = pulp.LpProblem("Supplier_Order_MOQ", pulp.LpMaximize)

# Integer order quantities x1, x2 and binary order indicator y
x1 = prob.add_variable("x1", lowBound=0, cat=pulp.LpInteger)
x2 = prob.add_variable("x2", lowBound=0, cat=pulp.LpInteger)
y  = prob.add_variable("y", cat=pulp.LpBinary)

prob += 7 * x1 + 5 * x2 - 12 * y, "Net_Profit"

prob += 2 * x1 + x2 <= 16, "Assembly_Hours"
prob += x1 + 2 * x2 <= 18, "Material_Limit"
prob += x2 <= 8, "Demand_Cap_B"
prob += x1 + x2 - 6 * y >= 0, "MOQ_Constraint"
prob += x1 - 18 * y <= 0, "Link_x1"
prob += x2 - 18 * y <= 0, "Link_x2"

res = prob.solve()
z_opt = pulp.value(prob.objective)
print(f"Status: {res.status.name}, Profit Z* = ${z_opt:.2f}")
print(f"y*={y.varValue:.0f}, x1*={x1.varValue:.0f}, x2*={x2.varValue:.0f}")

Welcome to the CBC MILP Solver 
Version: 2.10.13 
Build Date: Mar 11 2026 

command line - cbc /var/folders/x2/y620t7fd5dl06n7bq9jsb9qw0000gp/T/9e753290d68e4e29a9f0fb02c0c9d3dd-pulp.mps -timeMode elapsed -solve -printingOptions all -solution /var/folders/x2/y620t7fd5dl06n7bq9jsb9qw0000gp/T/9e753290d68e4e29a9f0fb02c0c9d3dd-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 11 COLUMNS
At line 33 RHS
At line 40 BOUNDS
At line 44 ENDATA
Problem MODEL has 6 rows, 3 columns and 12 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is -61.5556 - 0.00 seconds
Cgl0003I 0 fixed, 0 tightened bounds, 2 strengthened rows, 0 substitutions
Cgl0004I processed model has 5 rows, 3 columns (3 integer (1 of which binary)) and 13 elements
Cutoff increment increased from 1e-05 to 0.9999
Cbc0038I Initial state - 2 integers unsatisfied sum - 0.666667
Cbc0038I Solution found of -54
Cbc0038I Branch and bound need

#### 5b. Comparing Continuous vs. Integer Order Quantities in the MOQ Model

If $x_1, x_2 \ge 0$ are continuous, the intersection of $2x_1 + x_2 = 16$ and $x_1 + 2x_2 = 18$ gives $(x_1^*, x_2^*) = (14/3, 20/3) \approx (4.67, 6.67)$ with net profit $Z^* = \$54.00$. Restricting $x_1, x_2$ to integers yields $(x_1^*, x_2^*) = (5, 6)$ with net profit $Z^* = \$53.00$.

In [8]:
import pulp

prob_cont = pulp.LpProblem("Supplier_Order_MOQ_Continuous", pulp.LpMaximize)
x1_c = prob_cont.add_variable("x1", lowBound=0, cat=pulp.LpContinuous)
x2_c = prob_cont.add_variable("x2", lowBound=0, cat=pulp.LpContinuous)
y_c  = prob_cont.add_variable("y", cat=pulp.LpBinary)

prob_cont += 7 * x1_c + 5 * x2_c - 12 * y_c, "Net_Profit"
prob_cont += 2 * x1_c + x2_c <= 16, "Assembly_Hours"
prob_cont += x1_c + 2 * x2_c <= 18, "Material_Limit"
prob_cont += x2_c <= 8, "Demand_Cap_B"
prob_cont += x1_c + x2_c - 6 * y_c >= 0, "MOQ_Constraint"
prob_cont += x1_c - 18 * y_c <= 0, "Link_x1"
prob_cont += x2_c - 18 * y_c <= 0, "Link_x2"

prob_cont.solve(pulp.COIN_CMD(msg=False))
print(f"Continuous Quantities (y*={int(y_c.varValue)}): x1* = {x1_c.varValue:.2f}, x2* = {x2_c.varValue:.2f}, Net Profit Z* = ${pulp.value(prob_cont.objective):.2f}")
print(f"Integer Quantities    (y*=1): x1* = 5.00, x2* = 6.00, Net Profit Z* = $53.00")

Continuous Quantities (y*=1): x1* = 4.67, x2* = 6.67, Net Profit Z* = $54.00
Integer Quantities    (y*=1): x1* = 5.00, x2* = 6.00, Net Profit Z* = $53.00


### 6. Midterm Practice Problem: Product Mix, Shadow Prices & New Product Decisions

**Base Linear Program (Products A and B):**
$$\max Z = 30x_A + 20x_B$$
subject to:
$$\begin{aligned}
2x_A + 1x_B &\le 100 \quad (\text{Machine Hours } M) \\
1x_A + 1.5x_B &\le 90 \quad (\text{Labor Hours } L) \\
x_A, x_B &\ge 0
\end{aligned}$$

- **Optimal Primal Solution:** $x_A^\star = 30, \; x_B^\star = 40, \; Z^\star = \$1,700.00$
- **Unique Shadow Prices:** $y_M^\star = \$12.50/\text{hr}, \; y_L^\star = \$5.00/\text{hr}$
- **Candidate Product C** ($c_C = \$26$, uses $1\text{ M}, 2\text{ L}$):
  $$r_C = 26 - (12.50 \times 1 + 5.00 \times 2) = 26 - 22.50 = +\$3.50 > 0 \implies \text{Produce Product C!}$$
- **Candidate Product D** ($c_D = \$18$, uses $2\text{ M}, 1\text{ L}$):
  $$r_D = 18 - (12.50 \times 2 + 5.00 \times 1) = 18 - 30.00 = -\$12.00 \le 0 \implies \text{Do NOT Produce Product D!}$$

The cell below runs the exact `PuLP 4.0` code from **Section 7** of the slides (`Base_Product_Mix`).

In [9]:
import pulp

prob = pulp.LpProblem("Base_Product_Mix", pulp.LpMaximize)
xA = prob.add_variable("xA", lowBound=0, cat=pulp.LpContinuous)
xB = prob.add_variable("xB", lowBound=0, cat=pulp.LpContinuous)

prob += 30 * xA + 20 * xB, "Total_Profit"
prob += 2 * xA + 1 * xB <= 100, "Machine_Hours"
prob += 1 * xA + 1.5 * xB <= 90, "Labor_Hours"

res = prob.solve()
z_opt = pulp.value(prob.objective)
print(f"Status: {res.status.name}, Z* = ${z_opt:.2f}")
print(f"xA* = {xA.varValue:.1f}, xB* = {xB.varValue:.1f}")

# Extract Shadow Prices from prob.constraints() list in PuLP 4.0
duals = {c.name: abs(c.pi) for c in prob.constraints()}
print("Shadow Prices:", duals)
r_C = 26 - (1 * duals["Machine_Hours"] + 2 * duals["Labor_Hours"])
print(f"Reduced Cost of Product C: r_C = +${r_C:.2f}")

Welcome to the CBC MILP Solver 
Version: 2.10.13 
Build Date: Mar 11 2026 

command line - cbc /var/folders/x2/y620t7fd5dl06n7bq9jsb9qw0000gp/T/64d4c35c9eaf45eba301f350c1e328f8-pulp.mps -timeMode elapsed -solve -printingOptions all -solution /var/folders/x2/y620t7fd5dl06n7bq9jsb9qw0000gp/T/64d4c35c9eaf45eba301f350c1e328f8-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 7 COLUMNS
At line 14 RHS
At line 17 BOUNDS
At line 18 ENDATA
Problem MODEL has 2 rows, 2 columns and 4 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Presolve 2 (0) rows, 2 (0) columns and 4 (0) elements
0  Obj 0 Dual inf 50 (2)
0  Obj 0 Dual inf 50 (2)
2  Obj -1700
Optimal - objective value -1700
Optimal objective -1700 - 2 iterations time 0.002
Option for printingOptions changed from normal to all
Total time (CPU seconds):       0.00   (Wallclock seconds):       0.00

Status: Optimal, Z* = $1700.00
xA* = 30.0, xB* = 40.0
Shadow Prices: {'Ma

#### 6b. Re-Solving the Expanded LPs to Verify Reduced Cost & Shadow Price Predictions

Let's verify:
1. Adding **Product D** ($r_D = -\$12.00 \le 0$) leaves the optimal solution unchanged ($x_D^\star = 0, Z^\star = \$1,700.00$, and `xD.dj == -12.0`).
2. Adding **Product C** ($r_C = +\$3.50 > 0$) strictly increases total profit above $\$1,700.00$.
3. Increasing **Machine Hours** by $+5$ (from $100$ to $105$) increases optimal profit by $\Delta Z = 12.50 \times 5 = \$62.50$ to **$Z_{\text{new}}^\star = \$1,762.50$**.

In [10]:
import pulp

# 1. Verify Candidate Product D (Unprofitable: r_D = -$12.00)
prob_D = pulp.LpProblem("Product_Mix_With_D", pulp.LpMaximize)
xA = prob_D.add_variable("xA", lowBound=0)
xB = prob_D.add_variable("xB", lowBound=0)
xD = prob_D.add_variable("xD", lowBound=0)

prob_D += 30 * xA + 20 * xB + 18 * xD, "Total_Profit"
prob_D += 2 * xA + 1 * xB + 2 * xD <= 100, "Machine_Hours"
prob_D += 1 * xA + 1.5 * xB + 1 * xD <= 90, "Labor_Hours"
prob_D.solve(pulp.COIN_CMD(msg=False))
print(f"With Product D : Z* = ${pulp.value(prob_D.objective):,.2f} (xA*={xA.varValue:.1f}, xB*={xB.varValue:.1f}, xD*={xD.varValue:.1f}, Reduced Cost dj={xD.dj:.2f})")

# 2. Verify Candidate Product C (Profitable: r_C = +$3.50)
prob_C = pulp.LpProblem("Product_Mix_With_C", pulp.LpMaximize)
xA_c = prob_C.add_variable("xA", lowBound=0)
xB_c = prob_C.add_variable("xB", lowBound=0)
xC_c = prob_C.add_variable("xC", lowBound=0)

prob_C += 30 * xA_c + 20 * xB_c + 26 * xC_c, "Total_Profit"
prob_C += 2 * xA_c + 1 * xB_c + 1 * xC_c <= 100, "Machine_Hours"
prob_C += 1 * xA_c + 1.5 * xB_c + 2 * xC_c <= 90, "Labor_Hours"
prob_C.solve(pulp.COIN_CMD(msg=False))
print(f"With Product C : Z* = ${pulp.value(prob_C.objective):,.2f} (xA*={xA_c.varValue:.1f}, xB*={xB_c.varValue:.1f}, xC*={xC_c.varValue:.1f})")

# 3. Verify +5 Machine Hours (RHS_M = 105 -> Z_new = $1,762.50)
prob_rhs = pulp.LpProblem("Product_Mix_Plus_5M", pulp.LpMaximize)
xA_r = prob_rhs.add_variable("xA", lowBound=0)
xB_r = prob_rhs.add_variable("xB", lowBound=0)

prob_rhs += 30 * xA_r + 20 * xB_r, "Total_Profit"
prob_rhs += 2 * xA_r + 1 * xB_r <= 105, "Machine_Hours"
prob_rhs += 1 * xA_r + 1.5 * xB_r <= 90, "Labor_Hours"
prob_rhs.solve(pulp.COIN_CMD(msg=False))
print(f"+5 Machine Hrs : Z* = ${pulp.value(prob_rhs.objective):,.2f} (xA*={xA_r.varValue:.1f}, xB*={xB_r.varValue:.1f}, Delta Z = +${pulp.value(prob_rhs.objective) - 1700:.2f})")

With Product D : Z* = $1,700.00 (xA*=30.0, xB*=40.0, xD*=0.0, Reduced Cost dj=12.00)
With Product C : Z* = $1,793.33 (xA*=36.7, xB*=0.0, xC*=26.7)
+5 Machine Hrs : Z* = $1,762.50 (xA*=33.8, xB*=37.5, Delta Z = +$62.50)


### Q&A

- **Why can't we simply round the continuous LP relaxation solution to solve an Integer Linear Program (ILP)?**
  In Example 1 (`Simple_Pure_ILP`), the continuous LP relaxation optimum is $(x_1^*, x_2^*) = (1.33, 4.33)$ with $z_{LP}^* = 12.67$. Rounding to the nearest integer $(1, 4)$ gives $z = 11.00$, which is suboptimal, while rounding up violates the constraints. Branch-and-Bound identifies the true integer optimum at $(x_1^*, x_2^*) = (2, 3)$ with $z^* = 12.00$.
- **How do logical constraints and fixed setup costs change discrete optimization outcomes?**
  In the 4-item 0-1 Knapsack problem (Example 2), the unconstrained optimal selection is Laptop and Gold ($z^* = \$90$). Adding a mutual exclusion constraint between Laptop and Gold plus an implication constraint requiring Books whenever Gold is chosen (Example 3) shifts the optimal selection to Laptop and Silver ($z^* = \$70$).

### Data Analysis Key Findings

- **Pure ILP vs. LP Relaxation (Example 1):** Enforcing integrality reduces the maximum objective value from $z_{LP}^* = 12.67$ at $(1.33, 4.33)$ to $z_{ILP}^* = 12.00$ at $(2, 3)$.
- **0-1 Knapsack & Logical Constraints (Examples 2–3):** Under a $10\text{ kg}$ limit, packing Laptop ($4\text{ kg}, \$40$) and Gold ($3\text{ kg}, \$50$) achieves the maximum value of $\$90$. In the $6\text{ kg}$ counterexample (Example 2b), the greedy value-to-weight heuristic earns $\$32$, whereas the exact 0-1 ILP earns $\$44$.
- **Fixed Setup Cost & Minimum Order Quantity MILPs (Examples 4–5):** Using a tight Big-$M$ linking constraint ($x_1 + x_2 - 10y \le 0$) in Example 4 yields an optimal net profit of $Z^* = \$32.00$ at $(x_1^*, x_2^*, y^*) = (6.0, 4.0, 1)$. In the MOQ problem (Example 5), enforcing $x_1 + x_2 \ge 6y$ and $x_i \le 18y$ yields $Z^* = \$54.00$ at $(4.67, 6.67)$ for continuous quantities and $Z^* = \$53.00$ at $(5, 6)$ for integer quantities.
- **Sensitivity Analysis & New Product Pricing (Example 6):** For the base product mix ($x_A^\star = 30, x_B^\star = 40, Z^\star = \$1,700.00$), the shadow prices are $y_M^\star = \$12.50/\text{hr}$ and $y_L^\star = \$5.00/\text{hr}$. Candidate Product C has a positive reduced cost of $r_C = +\$3.50$ (increasing total profit to $\$1,770.00$ when introduced), whereas Candidate Product D has a negative reduced cost of $r_D = -\$12.00$. Adding $5$ Machine Hours increases optimal profit by $\$62.50$ to $\$1,762.50$.

### Insights or Next Steps

- **Always Verify Big-$M$ Bounds:** Choose the smallest valid upper bound $M$ implied by physical resource capacities (such as $M = 10$ in Example 4 and $M = 18$ in Example 5) to keep the LP relaxation tight and accelerate Branch-and-Bound convergence.
- **Sensitivity Analysis vs. Re-Optimization:** Use shadow prices and reduced costs for quick marginal screening of new products or capacity changes in continuous LPs, and re-solve the modified `PuLP` model whenever basis changes or integer restrictions apply.